In [ ]:
# import requests
# from datetime import datetime


# url = "https://ai-benchmark.com/data/results_phones.htm"
# headers = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
# }

# response = requests.get(url, headers=headers)
# response.raise_for_status()

# # Save raw HTML with current timestamp
# timestamp = datetime.now().strftime("ai-benchmark-%Y-%m-%d-%H-%M-%S")
# html_filename = f"{timestamp}.html"

# with open(html_filename, "w", encoding="utf-8") as file:
#     file.write(response.text)
# print(f"Raw HTML saved to {html_filename}\n")

Raw HTML saved to ai-benchmark-2026-04-20-15-47-53.html



In [17]:
import pandas as pd


file_path = "ai-benchmark-2026-04-20-15-47-53.html"
df = pd.read_html(file_path)[0]

phone_col = df.columns[df.iloc[1].str.contains(r'Phone\s+Model', regex=True, na=False)][0]
chipset_col = df.columns[df.iloc[1] == 'Chipset'].tolist()[0]

# print(df.iloc[5:][[phone_col, chipset_col]])
df

,0,1,2,3,4,5,6,7,8,9,...,100,101,102,103,104,105,106,107,108,109
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Phone Model,Chipset,Lib,AI Score,MobileNet-V3,MobileNet-V3,MobileNet-V3,MobileNet-V3,MobileNet-V3,...,MobileBERT,Llama2-42M,Llama2-42M,GPT-2,GPT-2,Stable Diffusion V1.5,Avg Init Time,Avg Init Time,ResNet-Memory,ResNet-Memory
2,NaN,Phone Model,Chipset,Lib,AI Score,CPU-Q,CPU-F,NN-INT8,NN-INT8,NN-INT8,...,NN-INT8,CPU-F,NN-INT8,CPU-F,NN-INT8,NN-INT8,NN-INT8,NN-FP16,NN-INT8,NN-FP16
3,NaN,Phone Model,Chipset,Lib,AI Score,ms,ms,ms,"ms, bs=4","error, L1",...,ms,ms,ms,ms,ms,ms,ms,ms,px,px
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
633,NaN,Samsung Galaxy Nexus,TI OMAP 4460,cc,10,1278,2692,1278,1314,0,...,5469,1724,1724,4707,4707,125029,51,N.A.,400,200
634,NaN,Asus Zenfone 2,Intel Atom Z3580,cc,5.8,13005,2171,13005,12886,0,...,38414,20585,20585,41371,41371,0,N.A.,N.A.,100,100
635,NaN,Sony Ericsson Xperia ray,Snapdragon S2,cc,4.7,1308,3117,1308,2205,0,...,9992,1959,1959,4991,4991,0,N.A.,N.A.,200,100
636,NaN,Samsung Galaxy Young,Snapdragon S1,cc,2.4,2837,5687,2837,2702,0,...,13363,4956,4956,0,0,283790,90,N.A.,100,100


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os 

os.makedirs("gpt-2", exist_ok=True)

file_path = "ai-benchmark-2026-04-20-15-47-53.html"
df = pd.read_html(file_path)[0]

phone_col = df.columns[df.iloc[1].str.contains(r'Phone\s+Model', regex=True, na=False)][0]
chipset_col = df.columns[df.iloc[1] == 'Chipset'].tolist()[0]

gpt2_cpu_f_col = None
for col in df.columns:
    if str(df.iloc[1, col]).strip() == 'GPT-2' and str(df.iloc[2, col]).strip() == 'CPU-F':
        gpt2_cpu_f_col = col
        break

if gpt2_cpu_f_col is not None:
    final_df = df.iloc[5:][[phone_col, chipset_col, gpt2_cpu_f_col]].copy()
    final_df.columns = ['device_name', 'chipset_name', 'compute_latency_ms']
    
    final_df['compute_latency_ms'] = pd.to_numeric(final_df['compute_latency_ms'], errors='coerce')
    final_df = final_df.dropna(subset=['compute_latency_ms']).reset_index(drop=True)
    
    # Plot before cleaning
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.boxplot(y=final_df['compute_latency_ms'])
    plt.title('Before Cleaning: Boxplot')
    plt.subplot(1, 2, 2)
    sns.histplot(final_df['compute_latency_ms'], bins=30, kde=True)
    plt.title('Before Cleaning: Distribution')
    plt.savefig('gpt-2/compute_trace_raw.png')
    plt.close()
    
    # 1. Filter out 0 ms
    cleaned_df = final_df[final_df['compute_latency_ms'] > 0].copy()
    
    # 2. Filter out upper outliers using IQR
    Q1 = cleaned_df['compute_latency_ms'].quantile(0.25)
    Q3 = cleaned_df['compute_latency_ms'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    
    cleaned_df = cleaned_df[cleaned_df['compute_latency_ms'] <= upper_bound].reset_index(drop=True)
    
    # Plot after cleaning
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.boxplot(y=cleaned_df['compute_latency_ms'])
    plt.title('After Cleaning: Boxplot')
    plt.subplot(1, 2, 2)
    sns.histplot(cleaned_df['compute_latency_ms'], bins=30, kde=True)
    plt.title('After Cleaning: Distribution')
    plt.savefig('gpt-2/compute_trace_cleaned.png')
    plt.close()
    
    print(f"Original Count: {len(final_df)}")
    print(f"Cleaned Count: {len(cleaned_df)}")
    print(f"Removed {len(final_df) - len(cleaned_df)} outliers/zeros.")
    
    # Save the cleaned dataframe
    cleaned_df.to_json("trace.json", orient="records", indent=4)
else:
    print("Could not locate the GPT-2 CPU-F column.")

Original Count: 633
Cleaned Count: 568
Removed 65 outliers/zeros.


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create the directory if it doesn't exist
os.makedirs("mobilenet-v3", exist_ok=True)

file_path = "ai-benchmark-2026-04-20-15-47-53.html"
df = pd.read_html(file_path)[0]

phone_col = df.columns[df.iloc[1].str.contains(r'Phone\s+Model', regex=True, na=False)][0]
chipset_col = df.columns[df.iloc[1] == 'Chipset'].tolist()[0]

mobilenetv3_cpu_q_col = None
for col in df.columns:
    if str(df.iloc[1, col]).strip() == 'MobileNet-V3' and str(df.iloc[2, col]).strip() == 'CPU-Q':
        mobilenetv3_cpu_q_col = col
        break

if mobilenetv3_cpu_q_col is not None:
    final_df = df.iloc[5:][[phone_col, chipset_col, mobilenetv3_cpu_q_col]].copy()
    final_df.columns = ['device_name', 'chipset_name', 'compute_latency_ms']
    
    final_df['compute_latency_ms'] = pd.to_numeric(final_df['compute_latency_ms'], errors='coerce')
    final_df = final_df.dropna(subset=['compute_latency_ms']).reset_index(drop=True)

    
    # Plot before cleaning
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.boxplot(y=final_df['compute_latency_ms'])
    plt.title('Before Cleaning: Boxplot')
    plt.subplot(1, 2, 2)
    sns.histplot(final_df['compute_latency_ms'], bins=30, kde=True)
    plt.title('Before Cleaning: Distribution')
    plt.savefig('mobilenet-v3/compute_trace_raw.png')
    plt.close()
    
    # 1. Filter out 0 ms
    cleaned_df = final_df[final_df['compute_latency_ms'] > 0].copy()
    
    # 2. Filter out upper outliers using IQR
    Q1 = cleaned_df['compute_latency_ms'].quantile(0.25)
    Q3 = cleaned_df['compute_latency_ms'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 3 * IQR
    
    cleaned_df = cleaned_df[cleaned_df['compute_latency_ms'] <= upper_bound].reset_index(drop=True)

    my_df = cleaned_df.copy()
    print(f"Compute: min={my_df['compute_latency_ms'].min()} ms, max={my_df['compute_latency_ms'].max()} ms, mean={my_df['compute_latency_ms'].mean():.2f} ms, median={my_df['compute_latency_ms'].median()} ms")
    
    
    # Plot after cleaning
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.boxplot(y=cleaned_df['compute_latency_ms'])
    plt.title('After Cleaning: Boxplot')
    plt.subplot(1, 2, 2)
    sns.histplot(cleaned_df['compute_latency_ms'], bins=30, kde=True)
    plt.title('After Cleaning: Distribution')
    plt.savefig('mobilenet-v3/compute_trace_cleaned.png')
    plt.close()
    
    print(f"Original Count: {len(final_df)}")
    print(f"Cleaned Count: {len(cleaned_df)}")
    print(f"Removed {len(final_df) - len(cleaned_df)} outliers/zeros.")
    
    # Save the cleaned dataframe
    cleaned_df.to_json("trace.json", orient="records", indent=4)
else:
    print("Could not locate the MobileNet-V3 CPU-F column.")

Compute: min=6 ms, max=471 ms, mean=108.31 ms, median=65.0 ms
Original Count: 633
Cleaned Count: 611
Removed 22 outliers/zeros.
